# Tutorial — Formatação de `datetime` com Python e Pandas

Este notebook apresenta, de forma progressiva e prática, como trabalhar com **datas e horários em Python e Pandas**.

Conteúdos:
- `datetime`, `date`, `time` e `timedelta`
- `strftime()` e `strptime()`
- `pd.to_datetime()`
- `.dt` para extrair componentes de datas
- tratamento de datas inválidas com `errors="coerce"`
- filtros por data
- `between()`
- `date_range()`
- `resample()`
- boas práticas para EDA e Data Science


In [ ]:
from datetime import datetime, date, time, timedelta
import pandas as pd


## 1. `datetime` no Python

`datetime` representa uma data combinada com horário.

In [ ]:
agora = datetime.now()
print(agora)


In [ ]:
data = datetime(2026, 10, 7)
print(data)

data_hora = datetime(2026, 10, 7, 14, 30, 45)
print(data_hora)


## 2. Extraindo componentes

Os principais atributos são `year`, `month`, `day`, `hour`, `minute` e `second`.

In [ ]:
data = datetime(2026, 10, 7, 14, 30, 45)

print("Ano:", data.year)
print("Mês:", data.month)
print("Dia:", data.day)
print("Hora:", data.hour)
print("Minuto:", data.minute)
print("Segundo:", data.second)
print("Dia da semana:", data.weekday())


## 3. `date` e `time`

Use `date` quando precisar apenas da data e `time` quando precisar apenas do horário.

In [ ]:
hoje = date.today()
print("Data:", hoje)

horario = time(14, 30, 45)
print("Horário:", horario)


## 4. `strftime()` — datetime → string

`strftime()` formata um objeto de data/hora em texto.

In [ ]:
data = datetime(2026, 10, 7, 14, 30, 45)

print(data.strftime("%d/%m/%Y"))
print(data.strftime("%d/%m/%Y %H:%M:%S"))
print(data.strftime("%H:%M"))


### Principais códigos do `strftime`

| Código | Significado | Exemplo |
|---|---|---|
| `%Y` | ano com 4 dígitos | `2026` |
| `%y` | ano com 2 dígitos | `26` |
| `%m` | mês | `10` |
| `%d` | dia | `07` |
| `%H` | hora (24h) | `14` |
| `%I` | hora (12h) | `02` |
| `%M` | minuto | `30` |
| `%S` | segundo | `45` |
| `%f` | microssegundos | `123456` |
| `%p` | AM/PM | `PM` |
| `%A` | dia da semana | `Wednesday` |
| `%a` | dia abreviado | `Wed` |
| `%B` | mês completo | `October` |
| `%b` | mês abreviado | `Oct` |


## 5. `strptime()` — string → datetime

`strptime()` faz o caminho inverso: transforma texto em um objeto `datetime`.

In [ ]:
data_texto = "07/10/2026"
data = datetime.strptime(data_texto, "%d/%m/%Y")

print(data)

data_texto = "07/10/2026 14:30:45"
data = datetime.strptime(data_texto, "%d/%m/%Y %H:%M:%S")

print(data)


### `strftime()` × `strptime()`

```text
datetime ──strftime()──> string
string   ──strptime()──> datetime
```

Uma forma de lembrar:
- `strftime`: **formatting**
- `strptime`: **parsing**


## 6. `timedelta`

`timedelta` representa intervalos de tempo e permite somar ou subtrair tempo.

In [ ]:
data = datetime(2026, 10, 7, 14, 0)

amanha = data + timedelta(days=1)
semana_passada = data - timedelta(days=7)
nova_data = data + timedelta(days=3, hours=2)

print("Original:", data)
print("Amanhã:", amanha)
print("Uma semana antes:", semana_passada)
print("3 dias e 2 horas depois:", nova_data)


## 7. Diferença entre duas datas

In [ ]:
inicio = datetime(2026, 10, 1)
fim = datetime(2026, 10, 7)

diferenca = fim - inicio

print(diferenca)
print("Dias:", diferenca.days)


# Parte 2 — Pandas

Em projetos de Data Science, uma situação muito comum é receber datas como texto e convertê-las para o tipo `datetime` do Pandas.

In [ ]:
df = pd.DataFrame({
    "data": [
        "01/10/2026",
        "02/10/2026",
        "03/10/2026"
    ]
})

print(df)
print()
print(df.dtypes)


## 8. `pd.to_datetime()`

Converta a coluna para `datetime64[ns]`.

> Boa prática: informe o `format` quando o formato dos dados for conhecido.

In [ ]:
df["data"] = pd.to_datetime(
    df["data"],
    format="%d/%m/%Y"
)

print(df)
print()
print(df.dtypes)


## 9. O acessor `.dt`

Depois que uma coluna possui tipo datetime, o Pandas disponibiliza o acessor `.dt` para trabalhar com seus componentes.

In [ ]:
df["ano"] = df["data"].dt.year
df["mes"] = df["data"].dt.month
df["dia"] = df["data"].dt.day
df["dia_semana"] = df["data"].dt.dayofweek
df["semana"] = df["data"].dt.isocalendar().week

df


## 10. Datas com horário

In [ ]:
df_hora = pd.DataFrame({
    "data_hora": [
        "07/10/2026 08:30:00",
        "07/10/2026 12:45:30",
        "08/10/2026 18:20:15"
    ]
})

df_hora["data_hora"] = pd.to_datetime(
    df_hora["data_hora"],
    format="%d/%m/%Y %H:%M:%S"
)

df_hora


In [ ]:
df_hora["data"] = df_hora["data_hora"].dt.date
df_hora["hora"] = df_hora["data_hora"].dt.hour
df_hora["minuto"] = df_hora["data_hora"].dt.minute
df_hora["segundo"] = df_hora["data_hora"].dt.second

df_hora


## 11. Formatando uma coluna Pandas

Use `.dt.strftime()` quando precisar **exibir** a data em um determinado formato.

⚠️ O resultado de `strftime()` é texto. Por isso, normalmente é melhor manter a coluna original como datetime.

In [ ]:
df["data_formatada"] = df["data"].dt.strftime("%d/%m/%Y")

print(df)
print()
print("Tipo da coluna original:", df["data"].dtype)
print("Tipo da coluna formatada:", df["data_formatada"].dtype)


## 12. `day_name()` e `month_name()`

In [ ]:
df_hora["dia_semana"] = df_hora["data_hora"].dt.day_name()
df_hora["mes_nome"] = df_hora["data_hora"].dt.month_name()

df_hora[["data_hora", "dia_semana", "mes_nome"]]


## 13. Traduzindo nomes de dias e meses

O Pandas retorna os nomes em inglês por padrão. Podemos usar um dicionário para traduzir.

In [ ]:
dias = {
    "Monday": "Segunda-feira",
    "Tuesday": "Terça-feira",
    "Wednesday": "Quarta-feira",
    "Thursday": "Quinta-feira",
    "Friday": "Sexta-feira",
    "Saturday": "Sábado",
    "Sunday": "Domingo"
}

meses = {
    "January": "Janeiro",
    "February": "Fevereiro",
    "March": "Março",
    "April": "Abril",
    "May": "Maio",
    "June": "Junho",
    "July": "Julho",
    "August": "Agosto",
    "September": "Setembro",
    "October": "Outubro",
    "November": "Novembro",
    "December": "Dezembro"
}

df_hora["dia_semana_pt"] = (
    df_hora["data_hora"].dt.day_name().map(dias)
)

df_hora["mes_nome_pt"] = (
    df_hora["data_hora"].dt.month_name().map(meses)
)

df_hora


## 14. Datas inválidas e `errors="coerce"`

Datas reais podem conter valores inválidos. Com `errors="coerce"`, valores que não podem ser convertidos tornam-se `NaT` (*Not a Time*).

In [ ]:
df_invalidas = pd.DataFrame({
    "data": [
        "07/10/2026",
        "08/10/2026",
        "data inválida",
        "10/10/2026"
    ]
})

df_invalidas["data"] = pd.to_datetime(
    df_invalidas["data"],
    format="%d/%m/%Y",
    errors="coerce"
)

df_invalidas


In [ ]:
print("Quantidade de datas inválidas:", df_invalidas["data"].isna().sum())

df_invalidas[df_invalidas["data"].isna()]


## 15. Filtrando datas

Com a coluna convertida para datetime, podemos comparar datas diretamente.

In [ ]:
df_hora[df_hora["data_hora"] >= "2026-10-07"]


In [ ]:
inicio = "2026-10-07"
fim = "2026-10-08"

df_hora[
    (df_hora["data_hora"] >= inicio) &
    (df_hora["data_hora"] < fim)
]


## 16. `between()`

Uma forma prática de filtrar um intervalo.

In [ ]:
df_hora[
    df_hora["data_hora"].between(
        "2026-10-07",
        "2026-10-08"
    )
]


## 17. `date_range()`

Cria uma sequência de datas.

In [ ]:
datas = pd.date_range(
    start="2026-01-01",
    end="2026-01-10"
)

print(datas)


In [ ]:
datas = pd.date_range(
    start="2026-01-01",
    periods=10,
    freq="D"
)

print(datas)


## 18. `resample()` — análise temporal

`resample()` é especialmente útil para séries temporais. Ele permite agregar dados por períodos como dia, semana, mês, trimestre ou ano.

In [ ]:
df_vendas = pd.DataFrame({
    "data": pd.date_range(
        "2026-01-01",
        periods=100,
        freq="D"
    ),
    "vendas": range(100)
})

df_vendas = df_vendas.set_index("data")

df_vendas.head()


In [ ]:
# Soma das vendas por mês
df_vendas["vendas"].resample("ME").sum()


In [ ]:
# Soma das vendas por semana
df_vendas["vendas"].resample("W").sum()


In [ ]:
# Soma das vendas por ano
df_vendas["vendas"].resample("YE").sum()


## 19. Exemplo completo de EDA temporal

Imagine um dataset de chamados. Vamos transformar a data em diferentes dimensões analíticas.

In [ ]:
df_chamados = pd.DataFrame({
    "data_abertura": [
        "01/10/2026 08:30:00",
        "01/10/2026 10:15:00",
        "02/10/2026 09:45:00",
        "03/10/2026 14:20:00"
    ],
    "chamados": [10, 15, 8, 20]
})

df_chamados["data_abertura"] = pd.to_datetime(
    df_chamados["data_abertura"],
    format="%d/%m/%Y %H:%M:%S"
)

df_chamados["ano"] = df_chamados["data_abertura"].dt.year
df_chamados["mes"] = df_chamados["data_abertura"].dt.month
df_chamados["dia"] = df_chamados["data_abertura"].dt.day
df_chamados["hora"] = df_chamados["data_abertura"].dt.hour
df_chamados["dia_semana"] = df_chamados["data_abertura"].dt.dayofweek

df_chamados


In [ ]:
print("Chamados por mês:")
print(df_chamados.groupby("mes")["chamados"].sum())

print("\nChamados por dia da semana:")
print(df_chamados.groupby("dia_semana")["chamados"].sum())

print("\nChamados por hora:")
print(df_chamados.groupby("hora")["chamados"].sum())


# 20. Boas práticas

### Durante a análise

Mantenha a coluna como:

```python
datetime64[ns]
```

Isso preserva a capacidade de:

- filtrar datas;
- ordenar cronologicamente;
- calcular diferenças;
- agrupar por períodos;
- usar `.dt`;
- usar `resample()`;
- trabalhar com séries temporais.

### Na apresentação

Use:

```python
df["data"].dt.strftime("%d/%m/%Y")
```

Assim, a data pode ser exibida no formato desejado sem destruir a coluna datetime original.

### Fluxo recomendado

```text
CSV / Excel / API
       ↓
    string
       ↓
pd.to_datetime()
       ↓
datetime64[ns]
       ↓
      .dt
       ↓
ano / mês / dia / hora
       ↓
EDA / filtros / groupby / resample
       ↓
visualização
       ↓
strftime()
       ↓
apresentação
```


# 21. Cheat Sheet

| Objetivo | Código |
|---|---|
| Data atual | `date.today()` |
| Data/hora atual | `datetime.now()` |
| Criar datetime | `datetime(2026, 10, 7)` |
| String → datetime | `datetime.strptime()` |
| datetime → string | `datetime.strftime()` |
| Pandas → datetime | `pd.to_datetime()` |
| Ano | `.dt.year` |
| Mês | `.dt.month` |
| Dia | `.dt.day` |
| Hora | `.dt.hour` |
| Minuto | `.dt.minute` |
| Segundo | `.dt.second` |
| Dia da semana | `.dt.dayofweek` |
| Nome do dia | `.dt.day_name()` |
| Nome do mês | `.dt.month_name()` |
| Diferença de datas | `data2 - data1` |
| Adicionar tempo | `+ timedelta()` |
| Remover tempo | `- timedelta()` |
| Datas inválidas → `NaT` | `errors="coerce"` |
| Filtrar intervalo | `.between()` |
| Criar sequência | `pd.date_range()` |
| Agrupar série temporal | `.resample()` |

---

## Exercício de fixação

Crie um DataFrame chamado `df_eventos` com pelo menos 10 registros contendo:

- `data_evento`
- `categoria`
- `quantidade`

Depois:

1. Converta `data_evento` para datetime.
2. Crie as colunas `ano`, `mes`, `dia` e `hora`.
3. Crie uma coluna com o nome do dia da semana.
4. Conte a quantidade total por mês.
5. Descubra qual dia da semana possui maior quantidade.
6. Filtre somente os eventos de um determinado mês.
7. Formate a data para `dd/mm/aaaa` apenas para apresentação.

### Desafio

Faça uma análise temporal utilizando `resample()` e apresente os resultados em um gráfico.
